In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from collections import defaultdict
from dataclasses import dataclass, asdict
from datetime import datetime
from functools import wraps
import csv
import heapq
import json
import math
import operator
import re
import tempfile
import time
import unittest
import uuid
from pathlib import Path

def _positive_number(value, label='amount'):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value <= 0:
        raise ValueError(f'{label} must be a positive finite number')


In [ ]:
def analyze_transactions(transactions):
    seen = set()
    totals = defaultdict(float)
    category_totals = defaultdict(lambda: defaultdict(float))
    customers = set()
    for tx in transactions:
        required = {'id', 'customer', 'category', 'amount', 'status'}
        if not isinstance(tx, dict) or not required <= tx.keys():
            raise ValueError('Each transaction must contain id, customer, category, amount, status')
        txid = tx['id']
        if txid in seen:
            raise ValueError(f'Duplicate transaction ID: {txid!r}')
        seen.add(txid)
        customer = tx['customer']
        if not isinstance(customer, str) or not customer.strip():
            raise ValueError('Customer name must be a non-empty string')
        customers.add(customer)
        amount = tx['amount']
        if isinstance(amount, bool) or not isinstance(amount, (int, float)) or not math.isfinite(amount) or amount < 0:
            raise ValueError('Transaction amount must be a non-negative finite number')
        if tx['status'] == 'success':
            totals[customer] += amount
            category_totals[customer][tx['category']] += amount
    ordered = sorted(customers, key=lambda c: (-totals[c], c))
    report = []
    for rank, customer in enumerate(ordered, 1):
        cats = category_totals[customer]
        top = sorted(cats, key=lambda c: (-cats[c], c))[0] if cats else None
        report.append({'customer': customer, 'total_spending': totals[customer], 'top_category': top, 'rank': rank})
    return report

assert [(r['customer'], r['total_spending']) for r in analyze_transactions([
    {'id': 1, 'customer': 'Ali', 'category': 'Food', 'amount': 500, 'status': 'success'},
    {'id': 2, 'customer': 'Ali', 'category': 'Travel', 'amount': 200, 'status': 'failed'},
    {'id': 3, 'customer': 'Sara', 'category': 'Food', 'amount': 700, 'status': 'success'}
])] == [('Sara', 700.0), ('Ali', 500.0)]

In [ ]:
def longest_unique_window(text):
    if not isinstance(text, str):
        raise TypeError('text must be a string')
    last = {}
    start = 0
    best_start = 0
    best_len = 0
    for i, char in enumerate(text):
        if char in last and last[char] >= start:
            start = last[char] + 1
        last[char] = i
        length = i - start + 1
        if length > best_len:
            best_start, best_len = start, length
    return text[best_start:best_start + best_len], best_start, best_len

assert longest_unique_window('abcabcbb') == ('abc', 0, 3)
assert longest_unique_window('') == ('', 0, 0)

In [ ]:
def find_paths(data, target):
    paths = []
    def equal(a, b):
        if isinstance(a, bool) != isinstance(b, bool) and isinstance(a, (bool, int, float)) and isinstance(b, (bool, int, float)):
            return False
        return type(a) == type(b) or (isinstance(a, (int, float)) and not isinstance(a, bool) and isinstance(b, (int, float)) and not isinstance(b, bool)) if a == b else False
    def walk(value, path):
        if equal(value, target):
            paths.append(path or '$')
        if isinstance(value, dict):
            for key, child in value.items():
                key_text = str(key)
                walk(child, f'{path}.{key_text}' if path else key_text)
        elif isinstance(value, list):
            for i, child in enumerate(value):
                walk(child, f'{path}[{i}]')
    walk(data, '')
    return paths

assert find_paths({'a': [{'x': 5}, 5], 'b': {'y': 5}}, 5) == ['a[0].x', 'a[1]', 'b.y']
assert find_paths({'x': True}, 1) == []

In [ ]:
def execution_order(tasks, dependencies):
    if len(set(tasks)) != len(tasks):
        raise ValueError('Duplicate task names are not allowed')
    task_set = set(tasks)
    graph = {task: [] for task in tasks}
    indegree = {task: 0 for task in tasks}
    for before, after in dependencies:
        if before not in task_set or after not in task_set:
            raise ValueError(f'Unknown task in dependency: {(before, after)!r}')
        graph[before].append(after)
        indegree[after] += 1
    ready = [task for task in tasks if indegree[task] == 0]
    heapq.heapify(ready)
    result = []
    while ready:
        task = heapq.heappop(ready)
        result.append(task)
        for nxt in graph[task]:
            indegree[nxt] -= 1
            if indegree[nxt] == 0:
                heapq.heappush(ready, nxt)
    if len(result) != len(tasks):
        raise ValueError('Dependency cycle detected')
    return result

assert execution_order(['build', 'test', 'deploy'], [('build', 'test'), ('test', 'deploy')]) == ['build', 'test', 'deploy']

In [ ]:
class ExpressionParser:
    def __init__(self, text):
        self.tokens = re.findall(r'\d+|[()+*/-]|\S', text)
        if ''.join(self.tokens).replace(' ', '') != re.sub(r'\s+', '', text):
            raise ValueError('Invalid expression')
        self.i = 0
    def peek(self): return self.tokens[self.i] if self.i < len(self.tokens) else None
    def take(self):
        token = self.peek()
        if token is not None: self.i += 1
        return token
    def parse(self):
        value = self.expr()
        if self.peek() is not None: raise ValueError('Unexpected token')
        return value
    def expr(self):
        value = self.term()
        while self.peek() in ('+', '-'):
            op = self.take(); rhs = self.term()
            value = value + rhs if op == '+' else value - rhs
        return value
    def term(self):
        value = self.unary()
        while self.peek() in ('*', '/'):
            op = self.take(); rhs = self.unary()
            if op == '/' and rhs == 0: raise ZeroDivisionError('division by zero')
            value = value * rhs if op == '*' else value / rhs
        return value
    def unary(self):
        if self.peek() == '+': self.take(); return self.unary()
        if self.peek() == '-': self.take(); return -self.unary()
        return self.atom()
    def atom(self):
        token = self.take()
        if token is None: raise ValueError('Expected a number or parenthesis')
        if token == '(':
            value = self.expr()
            if self.take() != ')': raise ValueError('Missing closing parenthesis')
            return value
        if token.isdigit(): return int(token)
        raise ValueError(f'Unexpected token: {token!r}')

def evaluate_expression(text):
    if not isinstance(text, str): raise TypeError('expression must be a string')
    return ExpressionParser(text).parse()

assert evaluate_expression('-(3 + 2) * 4 + 18 / 3') == -14.0
try:
    evaluate_expression('1/0')
except ZeroDivisionError:
    pass
else:
    raise AssertionError('Division by zero should fail')

In [ ]:
@dataclass(frozen=True)
class TransactionRecord:
    transaction_id: str
    action: str
    amount: float
    resulting_balance: float

class Account:
    def __init__(self, account_id, balance=0):
        if isinstance(balance, bool) or not isinstance(balance, (int, float)) or not math.isfinite(balance) or balance < 0:
            raise ValueError('Initial balance must be non-negative and finite')
        self.account_id = account_id
        self._balance = float(balance)
        self._transactions = []
    @property
    def balance(self): return self._balance
    @property
    def transactions(self): return tuple(self._transactions)
    def _record(self, action, amount):
        self._transactions.append(TransactionRecord(uuid.uuid4().hex, action, float(amount), self._balance))
    def deposit(self, amount):
        _positive_number(amount)
        self._balance += amount
        self._record('deposit', amount)
    def withdraw(self, amount):
        _positive_number(amount)
        if amount > self._balance: raise ValueError('Insufficient funds')
        self._balance -= amount
        self._record('withdrawal', amount)
    def statement(self): return list(self._transactions)

class Bank:
    def __init__(self): self.accounts = {}
    def add_account(self, account):
        if account.account_id in self.accounts: raise ValueError('Duplicate account ID')
        self.accounts[account.account_id] = account
    def _get(self, account_id):
        if account_id not in self.accounts: raise KeyError(f'Account not found: {account_id}')
        return self.accounts[account_id]
    def transfer(self, source_id, destination_id, amount):
        _positive_number(amount)
        source, dest = self._get(source_id), self._get(destination_id)
        if source is dest: raise ValueError('Cannot transfer to the same account')
        if source.balance < amount: raise ValueError('Insufficient funds')
        # Validate everything before changing either account.
        source._balance -= amount; dest._balance += amount
        source._record(f'transfer_out:{destination_id}', amount)
        dest._record(f'transfer_in:{source_id}', amount)

a, b = Account('A', 1000), Account('B', 200)
bank = Bank(); bank.add_account(a); bank.add_account(b); bank.transfer('A', 'B', 350)
assert (a.balance, b.balance) == (650, 550)

In [ ]:
class Employee(ABC):
    def __init__(self, name): self.name = name
    @abstractmethod
    def calculate_pay(self): ...

class SalariedEmployee(Employee):
    def __init__(self, name, salary): super().__init__(name); self.salary = self._valid(salary)
    @staticmethod
    def _valid(v):
        if isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) or v < 0: raise ValueError('Pay inputs must be non-negative finite numbers')
        return float(v)
    def calculate_pay(self): return self.salary

class HourlyEmployee(Employee):
    def __init__(self, name, rate, hours):
        super().__init__(name); self.rate = SalariedEmployee._valid(rate); self.hours = SalariedEmployee._valid(hours)
    def calculate_pay(self): return min(self.hours, 40) * self.rate + max(0, self.hours - 40) * self.rate * 1.5

class CommissionEmployee(Employee):
    def __init__(self, name, base, sales, rate):
        super().__init__(name); self.base = SalariedEmployee._valid(base); self.sales = SalariedEmployee._valid(sales); self.rate = SalariedEmployee._valid(rate)
    def calculate_pay(self): return self.base + self.sales * self.rate

def payroll_summary(employees):
    return [{'name': e.name, 'pay': e.calculate_pay()} for e in employees]

assert HourlyEmployee('H', 20, 45).calculate_pay() == 950
assert CommissionEmployee('C', 500, 4000, .10).calculate_pay() == 900

In [ ]:
def ttl_cache(seconds):
    if isinstance(seconds, bool) or not isinstance(seconds, (int, float)) or seconds < 0: raise ValueError('seconds must be non-negative')
    def decorate(func):
        cache = {}
        stats = {'hits': 0, 'misses': 0}
        @wraps(func)
        def wrapper(*args, **kwargs):
            try:
                key = (args, tuple(sorted(kwargs.items())))
                hash(key)
            except TypeError as exc:
                raise TypeError('ttl_cache arguments must be hashable') from exc
            now = time.monotonic()
            entry = cache.get(key)
            if entry is not None and now < entry[0]:
                stats['hits'] += 1; return entry[1]
            stats['misses'] += 1
            value = func(*args, **kwargs)
            cache[key] = (now + seconds, value)
            return value
        wrapper.cache_info = lambda: dict(stats)
        wrapper.cache_clear = lambda: (cache.clear(), stats.update(hits=0, misses=0))
        return wrapper
    return decorate

@ttl_cache(10)
def add(x, y): return x + y
assert add(x=2, y=3) == add(y=3, x=2)
assert add.cache_info()['hits'] == 1

In [ ]:
LOG_PATTERN = re.compile(r'^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\s*\|\s*([A-Za-z]+)\s*\|\s*(.*)$')

def read_errors(path, malformed_counter=None):
    count = malformed_counter if malformed_counter is not None else {'count': 0}
    with open(path, 'r', encoding='utf-8') as file:
        for line_number, line in enumerate(file, 1):
            match = LOG_PATTERN.match(line.rstrip('\n'))
            if not match:
                count['count'] += 1; continue
            timestamp, level, message = match.groups()
            try: datetime.strptime(timestamp, '%Y-%m-%d %H:%M:%S')
            except ValueError:
                count['count'] += 1; continue
            if level.upper() == 'ERROR': yield line_number, timestamp, message

def summarize_errors(path, malformed_counter=None):
    counts = defaultdict(int)
    for _, timestamp, _ in read_errors(path, malformed_counter): counts[timestamp[:10]] += 1
    return dict(sorted(counts.items()))

with tempfile.TemporaryDirectory() as d:
    p = Path(d) / 'app.log'; p.write_text('2026-11-07 10:30:00 | ERROR | Connection lost\n', encoding='utf-8')
    assert list(read_errors(p)) == [(1, '2026-11-07 10:30:00', 'Connection lost')]

In [ ]:
def reconcile_inventory(inventory_path, movements_path, updated_path, rejected_path):
    stock = {}
    names = {}
    with open(inventory_path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            sku = row['sku'].strip()
            try: qty = int(row['quantity'])
            except (ValueError, TypeError): raise ValueError(f'Invalid quantity for SKU {sku}')
            if not sku or qty < 0 or sku in stock: raise ValueError(f'Invalid or duplicate inventory SKU: {sku}')
            stock[sku] = qty; names[sku] = row['name']
    rejected = []
    with open(movements_path, newline='', encoding='utf-8') as f:
        for row_num, row in enumerate(csv.DictReader(f), 2):
            sku = row.get('sku', '').strip(); kind = row.get('type', '').strip().upper()
            try: units = int(row.get('units', ''))
            except (ValueError, TypeError): units = 0
            reason = None
            if sku not in stock: reason = 'Unknown SKU'
            elif units <= 0: reason = 'Units must be positive'
            elif kind not in ('IN', 'OUT'): reason = 'Invalid movement type'
            elif kind == 'OUT' and stock[sku] - units < 0: reason = 'Insufficient stock'
            if reason: rejected.append({**row, 'row_number': row_num, 'reason': reason}); continue
            stock[sku] += units if kind == 'IN' else -units
    with open(updated_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['sku', 'name', 'quantity']); writer.writeheader()
        for sku in stock: writer.writerow({'sku': sku, 'name': names[sku], 'quantity': stock[sku]})
    fields = list(rejected[0].keys()) if rejected else ['row_number', 'sku', 'type', 'units', 'reason']
    with open(rejected_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fields); writer.writeheader(); writer.writerows(rejected)
    return stock, rejected

In [ ]:
def build_gradebook(students_path, scores_path, report_path):
    students = {}
    with open(students_path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            sid = row['student_id'].strip()
            if not sid or sid in students: raise ValueError(f'Invalid or duplicate student ID: {sid}')
            students[sid] = row['name']
    scores = defaultdict(list); seen = set()
    with open(scores_path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            sid, subject = row['student_id'].strip(), row['subject'].strip()
            if sid not in students: raise ValueError(f'Unknown student ID: {sid}')
            key = (sid, subject)
            if key in seen: raise ValueError(f'Duplicate student-subject entry: {key}')
            seen.add(key)
            try: mark = float(row['marks'])
            except (ValueError, TypeError): raise ValueError('Marks must be numeric')
            if not math.isfinite(mark) or not 0 <= mark <= 100: raise ValueError('Marks must be between 0 and 100')
            scores[sid].append(mark)
    def grade(avg): return 'A' if avg >= 85 else 'B' if avg >= 70 else 'C' if avg >= 55 else 'D' if avg >= 40 else 'F'
    rows = []
    for sid, name in students.items():
        avg = sum(scores[sid]) / len(scores[sid]) if scores[sid] else 0.0
        rows.append({'student_id': sid, 'name': name, 'average': round(avg, 2), 'grade': grade(avg)})
    rows.sort(key=lambda r: (-r['average'], r['student_id']))
    for rank, row in enumerate(rows, 1): row['rank'] = rank
    with open(report_path, 'w', newline='', encoding='utf-8') as f:
        fields = ['student_id', 'name', 'average', 'grade', 'rank']; writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader(); writer.writerows(rows)
    return rows

In [ ]:
class ReversibleRange:
    def __init__(self, start, stop, step=1):
        for name, value in [('start', start), ('stop', stop), ('step', step)]:
            if isinstance(value, bool) or not isinstance(value, int): raise TypeError(f'{name} must be an integer')
        if step == 0: raise ValueError('step must not be zero')
        self.start, self.stop, self.step = start, stop, step
    def __iter__(self):
        current = self.start
        if self.step > 0:
            while current < self.stop: yield current; current += self.step
        else:
            while current > self.stop: yield current; current += self.step
    def __len__(self):
        if self.step > 0: return max(0, (self.stop - self.start + self.step - 1) // self.step)
        return max(0, (self.start - self.stop - self.step - 1) // (-self.step))
    def __reversed__(self):
        n = len(self)
        for i in range(n - 1, -1, -1): yield self.start + i * self.step
    def __contains__(self, value):
        if isinstance(value, bool) or not isinstance(value, int): return False
        if self.step > 0 and not self.start <= value < self.stop: return False
        if self.step < 0 and not self.stop < value <= self.start: return False
        return (value - self.start) % self.step == 0

r = ReversibleRange(2, 11, 3)
assert list(r) == [2, 5, 8] and list(reversed(r)) == [8, 5, 2] and 5 in r

In [ ]:
class BookingScheduler:
    def __init__(self): self.bookings = []; self.ids = set()
    def add_booking(self, room, start, end, booking_id):
        if not isinstance(start, datetime) or not isinstance(end, datetime): raise TypeError('start and end must be datetime objects')
        if end <= start: return {'accepted': False, 'reason': 'End must be after start'}
        if booking_id in self.ids: return {'accepted': False, 'reason': 'Duplicate booking ID'}
        for b in self.bookings:
            if b['room'] == room and start < b['end'] and b['start'] < end:
                return {'accepted': False, 'reason': 'Booking overlaps an existing booking', 'conflict': b['booking_id']}
        booking = {'room': room, 'start': start, 'end': end, 'booking_id': booking_id}
        self.bookings.append(booking); self.ids.add(booking_id)
        self.bookings.sort(key=lambda b: (b['room'], b['start'], b['booking_id']))
        return {'accepted': True, 'booking': booking}

scheduler = BookingScheduler()
d = datetime(2026, 11, 7)
assert scheduler.add_booking('A', d.replace(hour=9), d.replace(hour=10), '1')['accepted']
assert scheduler.add_booking('A', d.replace(hour=10), d.replace(hour=11), '2')['accepted']
assert not scheduler.add_booking('A', d.replace(hour=9, minute=30), d.replace(hour=10, minute=30), '3')['accepted']

In [ ]:
def load_contacts(path):
    path = Path(path)
    if not path.exists(): return {}
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
        if not isinstance(data, dict): raise ValueError('Contact data must be a JSON object')
        return data
    except (json.JSONDecodeError, UnicodeDecodeError) as exc:
        # Preserve corrupt source file; raise instead of overwriting it silently.
        raise ValueError(f'Contact file is corrupt: {path}') from exc

def save_contacts(path, contacts):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(contacts, indent=2), encoding='utf-8')
    temp.replace(path)

def add_contact(contacts, contact_id, name, phone, email):
    contact_id = str(contact_id)
    if contact_id in contacts: raise ValueError('Contact ID already exists')
    if not all(isinstance(v, str) and v.strip() for v in (name, phone, email)): raise ValueError('Name, phone, and email are required')
    contacts[contact_id] = {'id': contact_id, 'name': name.strip(), 'phone': phone.strip(), 'email': email.strip()}
    return contacts[contact_id]

def search_contacts(contacts, query):
    q = query.casefold()
    return [c for c in contacts.values() if q in c['name'].casefold()]

def update_contact(contacts, contact_id, **changes):
    contact_id = str(contact_id)
    if contact_id not in contacts: raise KeyError('Contact not found')
    for field in ('name', 'phone', 'email'):
        if field in changes:
            if not isinstance(changes[field], str) or not changes[field].strip(): raise ValueError(f'{field} cannot be empty')
            contacts[contact_id][field] = changes[field].strip()
    return contacts[contact_id]

def delete_contact(contacts, contact_id):
    try: return contacts.pop(str(contact_id))
    except KeyError as exc: raise KeyError('Contact not found') from exc

def contact_manager_cli(path='contacts.json'):
    contacts = load_contacts(path)
    while True:
        print('\n1 Add  2 Search  3 Update  4 Delete  5 List  0 Exit')
        choice = input('Choose: ').strip()
        try:
            if choice == '1': add_contact(contacts, input('ID: '), input('Name: '), input('Phone: '), input('Email: ')); save_contacts(path, contacts)
            elif choice == '2': print(search_contacts(contacts, input('Name search: ')))
            elif choice == '3': update_contact(contacts, input('ID: '), name=input('New name: ')); save_contacts(path, contacts)
            elif choice == '4': delete_contact(contacts, input('ID: ')); save_contacts(path, contacts)
            elif choice == '5': print(list(contacts.values()))
            elif choice == '0': break
            else: print('Unknown option')
        except (ValueError, KeyError) as exc: print(f'Error: {exc}')

# Run manually with: contact_manager_cli()

In [ ]:
@dataclass
class Product:
    product_id: str
    name: str
    price: float
    stock: int
    def __post_init__(self):
        if not self.product_id or not self.name: raise ValueError('Product ID and name are required')
        if isinstance(self.price, bool) or not isinstance(self.price, (int, float)) or not math.isfinite(self.price) or self.price < 0: raise ValueError('Price must be non-negative and finite')
        if isinstance(self.stock, bool) or not isinstance(self.stock, int) or self.stock < 0: raise ValueError('Stock must be a non-negative integer')

class Inventory:
    def __init__(self, products):
        self.products = {p.product_id: p for p in products}
        if len(self.products) != len(products): raise ValueError('Duplicate product ID')
    @classmethod
    def from_json(cls, path):
        data = json.loads(Path(path).read_text(encoding='utf-8'))
        return cls([Product(str(p['product_id']), p['name'], p['price'], p['stock']) for p in data])

class Cart:
    def __init__(self): self.items = defaultdict(int)
    def add(self, product_id, quantity):
        if isinstance(quantity, bool) or not isinstance(quantity, int) or quantity <= 0: raise ValueError('Quantity must be a positive integer')
        self.items[product_id] += quantity
    def total(self, inventory, discount_percent=0):
        if isinstance(discount_percent, bool) or not isinstance(discount_percent, (int, float)) or not 0 <= discount_percent <= 100: raise ValueError('Discount must be between 0 and 100')
        subtotal = 0
        for pid, qty in self.items.items():
            if pid not in inventory.products: raise KeyError(f'Unknown product: {pid}')
            subtotal += inventory.products[pid].price * qty
        return round(subtotal * (1 - discount_percent / 100), 2)

class Order:
    def __init__(self, order_id, items, total): self.order_id, self.items, self.total = order_id, dict(items), total
    def to_dict(self): return {'order_id': self.order_id, 'items': self.items, 'total': self.total}

def place_order(cart, inventory, orders_path, discount_percent=0):
    if not cart.items: raise ValueError('Cannot place an empty order')
    # Validate every item before modifying stock.
    for pid, qty in cart.items.items():
        if pid not in inventory.products: raise KeyError(f'Unknown product: {pid}')
        if qty <= 0 or inventory.products[pid].stock < qty: raise ValueError(f'Invalid quantity or insufficient stock for {pid}')
    total = cart.total(inventory, discount_percent)
    order = Order(uuid.uuid4().hex, cart.items, total)
    path = Path(orders_path)
    old_stock = {pid: inventory.products[pid].stock for pid in cart.items}
    for pid, qty in cart.items.items(): inventory.products[pid].stock -= qty
    try:
        existing = []
        if path.exists():
            existing = json.loads(path.read_text(encoding='utf-8'))
            if not isinstance(existing, list): raise ValueError('Orders file must contain a JSON list')
        existing.append(order.to_dict())
        path.parent.mkdir(parents=True, exist_ok=True)
        temp = path.with_suffix(path.suffix + '.tmp')
        temp.write_text(json.dumps(existing, indent=2), encoding='utf-8')
        temp.replace(path)
    except Exception:
        for pid, qty in old_stock.items(): inventory.products[pid].stock = qty
        raise
    return order

def load_products(path):
    data = json.loads(Path(path).read_text(encoding='utf-8'))
    return [Product(str(p['product_id']), p['name'], p['price'], p['stock']) for p in data]

# Three basic unit tests for Task 15.
class OrderSystemTests(unittest.TestCase):
    def setUp(self):
        self.inventory = Inventory([Product('P1', 'Demo', 100, 5)])
        self.tempdir = tempfile.TemporaryDirectory()
        self.orders_path = Path(self.tempdir.name) / 'orders.json'
    def tearDown(self): self.tempdir.cleanup()
    def test_discount_and_stock_update(self):
        cart = Cart(); cart.add('P1', 2)
        order = place_order(cart, self.inventory, self.orders_path, 10)
        self.assertEqual(order.total, 180); self.assertEqual(self.inventory.products['P1'].stock, 3)
    def test_out_of_stock_rejected_without_changes(self):
        cart = Cart(); cart.add('P1', 6)
        with self.assertRaises(ValueError): place_order(cart, self.inventory, self.orders_path)
        self.assertEqual(self.inventory.products['P1'].stock, 5)
    def test_invalid_quantity_rejected(self):
        cart = Cart()
        with self.assertRaises(ValueError): cart.add('P1', 0)

unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(OrderSystemTests))

test_discount_and_stock_update (__main__.OrderSystemTests.test_discount_and_stock_update) ... ok
test_invalid_quantity_rejected (__main__.OrderSystemTests.test_invalid_quantity_rejected) ... ok
test_out_of_stock_rejected_without_changes (__main__.OrderSystemTests.test_out_of_stock_rejected_without_changes) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.010s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>